<a href="https://colab.research.google.com/github/kpakshaya123/Rynix-Internship/blob/main/Sprint1(Intern_dataset).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Intern Performance Analytics & Insights Dashboard

In [58]:
import pandas as pd
df=pd.read_csv('Internship dataset uncleaned.csv')

In [59]:
df.shape

(53, 26)

Remove Duplicates based on the InternId

In [60]:
df = df.drop_duplicates(subset='Internid').reset_index(drop=True)

In [61]:
df.shape

(51, 26)

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Internid               51 non-null     object 
 1   Name                   51 non-null     object 
 2   Email id               51 non-null     object 
 3   Gender                 51 non-null     object 
 4   Phone number           51 non-null     int64  
 5   Domain                 51 non-null     object 
 6   Date Range             51 non-null     object 
 7   Total days             51 non-null     int64  
 8   sod filled             51 non-null     int64  
 9   eod filled             51 non-null     int64  
 10  Attendance             33 non-null     float64
 11  Total Task             51 non-null     int64  
 12  Task1 completed        51 non-null     object 
 13  Task1 score            51 non-null     int64  
 14  Task2 completed        51 non-null     object 
 15  Task2 sc

In [63]:
df.describe()

,Phone number,Total days,sod filled,eod filled,Attendance,Total Task,Task1 score,Task2 score,Task3 score,Task4 score,Count overall task,Task completion,performance score
count,5.100000e+01,51.0,51.000000,51.000000,33.000000,51.0,51.000000,51.00000,51.000000,51.000000,51.000000,47.000000,41.000000
mean,9.876543e+09,31.0,25.431373,25.058824,79.575758,4.0,70.176471,66.45098,70.196078,68.843137,3.196078,82.446809,66.707317
std,1.449205e+01,0.0,7.088737,7.425394,23.693393,0.0,36.172203,32.38414,33.918738,37.549633,1.385924,31.672103,32.867342
min,9.876543e+09,31.0,10.000000,10.000000,34.000000,4.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,9.876543e+09,31.0,21.000000,20.500000,66.000000,4.0,69.000000,68.00000,57.500000,65.000000,3.000000,75.000000,33.000000
50%,9.876543e+09,31.0,29.000000,29.000000,90.000000,4.0,88.000000,81.00000,86.000000,88.000000,4.000000,100.000000,84.000000
75%,9.876543e+09,31.0,31.000000,31.000000,98.000000,4.0,94.000000,86.00000,93.500000,94.000000,4.000000,100.000000,90.000000
max,9.876543e+09,31.0,31.000000,31.000000,100.000000,4.0,99.000000,96.00000,98.000000,100.000000,4.000000,100.000000,98.000000


Trim the Intern Id and Name


In [64]:
df['Internid']=df['Internid'].astype(str).str.strip()
df['Name']=df['Name'].str.strip()

Replace the Incorrect Values as (at) as @

In [65]:
df.columns = df.columns.str.strip()
df['Email id'] = df['Email id'].str.replace('(at)', '@', regex=False)

CapitaliZe the First Letter

In [66]:
df['Gender'] = df['Gender'].str.strip().str.capitalize()

Capitalize the first letter

In [67]:
df['Domain']=df['Domain'].str.strip().str.capitalize()

Calculating Attendance Percentage

In [68]:
calculated_val = (((df['sod filled'].fillna(0) + df['eod filled'].fillna(0)) / 2) / 31 * 100).round(0)
df['Attendance'] = df['Attendance'].fillna(calculated_val).fillna(0).astype(int)

Calculate Count over all task they are completed

In [69]:
df['Count overall task'] = (
    (df[['Task1 completed', 'Task2 completed', 'Task3 completed', 'Task4 completed']] == "Yes")
    .sum(axis=1)
)

Task Completion out of 4 task

In [70]:
df.columns = df.columns.str.strip()
df['Task completion'] = ((df['Count overall task'] / 4) * 100).astype(int)

Calculating Task Completion percentage (Missing value in a performance score)

In [71]:
df['performance score'] = df['performance score'].fillna(
    df['Task1 score'] +
    df['Task2 score'] +
    df['Task3 score'] +
    df['Task4 score']
/ 4).round(0).astype(int)

In [72]:
def get_status(score):
    if score >= 90:
        return "Outstanding"
    elif score >= 70:
        return "Excellent"
    elif score >= 65:
        return "Good"
    elif score >= 40:
        return "Average"
    else:
        return "Improvement"

df["performance status"] = df["performance score"].apply(get_status)

Calculating Stipend based on (performance score, attendance and task completion)            
Missing Values in Stipend


In [73]:
def calculate_Stipend(row):
    if (
        row['performance score'] >= 80 and
        row['Attendance'] >= 80 and
        row['Task completion'] >= 80
    ):
        return 15000

    elif (
        row['performance score'] >= 50 and
        row['Attendance'] >= 50 and
        row['Task completion'] >= 50
    ):
        return 10000

    else:
        return "0"

df['Stipend'] = df.apply(calculate_Stipend, axis=1)

Hiring Recommendation is Calculated Based on the Stipend

In [74]:
def hiring_recommendation(stipend):
    if stipend == 15000:
        return "Hired as Employee"
    elif stipend == 10000:
        return "Intern for 6 months"
    else:
        return "Intern for 1 month"

df['Hiring Recommendation'] = df['Stipend'].apply(hiring_recommendation)

In [75]:
df.to_csv("cleaned Internship dataset.csv", index=False)